<a href="https://colab.research.google.com/github/bumsootead/Seoul_Subway_Analytics/blob/main/python_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Mobility Analysis and Station Classification

## Questions
1. Which stations and lines have the highest activity?
2. How do weekday and weekend patterns differ?
3. Which stations behave like employment, residential, commercial,
   or mixed-use mobility hubs?
4. How did stations change between Jul–Dec 2024 and Jul–Dec 2025?
5. How do 2025 passenger types vary by time and location?

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="deep")

CLEANED_DIR = Path("/content/")
OUTPUT_DIR = Path("outputs/figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

daily = pd.read_csv(r"C:\cleaned\station_daily_wide.csv")

daily.head()

,service_date,source_year,line_name,station_code,station_name,day_of_week,is_weekday,is_weekend,month,year_month,alightings,boardings,total_activity,net_boarding
0,2024-01-01,2024,1호선,150,서울역,Monday,True,False,1,2024-01,31810,36085,67895,4275
1,2024-01-01,2024,1호선,151,시청,Monday,True,False,1,2024-01,9704,11029,20733,1325
2,2024-01-01,2024,1호선,152,종각,Monday,True,False,1,2024-01,13191,16379,29570,3188
3,2024-01-01,2024,1호선,153,종로3가,Monday,True,False,1,2024-01,11650,14309,25959,2659
4,2024-01-01,2024,1호선,154,종로5가,Monday,True,False,1,2024-01,10180,11200,21380,1020


In [13]:
daily = pd.read_csv(
    CLEANED_DIR "C:\cleaned\station_daily_wide.csv",
    encoding="utf-8-sig",
    parse_dates=["service_date"]
)

# Load the total hourly dataset for hour-by-hour / peak-time analysis.
hourly = pd.read_csv(
    CLEANED_DIR  "C:\cleaned\station_hourly_total.csv",
    encoding="utf-8-sig",
    parse_dates=["service_date"]
)

# Load the passenger-type dataset for 2025 passenger analysis.
passenger_type_2025 = pd.read_csv(
    CLEANED_DIR "C:\cleaned\station_hourly_passenger_type_2025.csv",
    encoding="utf-8-sig",
    parse_dates=["service_date"]
)

daily.head()

<>:2: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:9: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:16: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:2: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:9: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:16: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
C:\Users\ryanj\AppData\Local\Temp\ipykernel_19364\3400189438.py:2: SyntaxWarning

,service_date,source_year,line_name,station_code,station_name,day_of_week,is_weekday,is_weekend,month,year_month,alightings,boardings,total_activity,net_boarding
0,2024-01-01,2024,1호선,150,서울역,Monday,True,False,1,2024-01,31810,36085,67895,4275
1,2024-01-01,2024,1호선,151,시청,Monday,True,False,1,2024-01,9704,11029,20733,1325
2,2024-01-01,2024,1호선,152,종각,Monday,True,False,1,2024-01,13191,16379,29570,3188
3,2024-01-01,2024,1호선,153,종로3가,Monday,True,False,1,2024-01,11650,14309,25959,2659
4,2024-01-01,2024,1호선,154,종로5가,Monday,True,False,1,2024-01,10180,11200,21380,1020


In [15]:
top_10_total = (
    daily.groupby(["station_name", "line_name"], as_index=False)
    ["total_activity"]
    .sum()
    .sort_values("total_activity", ascending=False)
    .head(10)
)

top_10_weekday = (
    daily[daily["is_weekday"]]
    .groupby(["station_name", "line_name"], as_index=False)
    ["total_activity"]
    .mean()
    .sort_values("total_activity", ascending=False)
    .head(10)
)

top_10_weekend = (
    daily[daily["is_weekend"]]
    .groupby(["station_name", "line_name"], as_index=False)
    ["total_activity"]
    .mean()
    .sort_values("total_activity", ascending=False)
    .head(10)
)

busiest_lines = (
    daily.groupby("line_name", as_index=False)
    ["total_activity"]
    .agg(["sum", "mean"])
    .reset_index()
)

# Create station-function measures

In [18]:
morning_hours = ["06_07", "07_08", "08_09"]
evening_hours = ["17_18", "18_19", "19_20"]
midday_hours = ["10_11", "11_12", "12_13", "13_14", "14_15"]

weekday_hourly = hourly[hourly["is_weekday"]].copy()

station_direction = (
    weekday_hourly
    .groupby(
        [
            "source_year", "line_name", "station_code",
            "station_name", "direction", "hour_bucket"
        ],
        as_index=False
    )["passenger_count"]
    .sum()
)

station_profile = (
    station_direction
    .pivot_table(
        index=[
            "source_year", "line_name",
            "station_code", "station_name"
        ],
        columns=["direction", "hour_bucket"],
        values="passenger_count",
        aggfunc="sum",
        fill_value=0
    )
)

def period_total(frame, direction, hour_list):
    return frame[
        [
            (direction, hour)
            for hour in hour_list
            if (direction, hour) in frame.columns
        ]
    ].sum(axis=1)

station_profile["morning_boardings"] = period_total(
    station_profile, "board", morning_hours
)
station_profile["morning_alightings"] = period_total(
    station_profile, "alight", morning_hours
)
station_profile["evening_boardings"] = period_total(
    station_profile, "board", evening_hours
)
station_profile["evening_alightings"] = period_total(
    station_profile, "alight", evening_hours
)

station_profile["midday_activity"] = (
    period_total(station_profile, "board", midday_hours)
    + period_total(station_profile, "alight", midday_hours)
)

station_profile["total_weekday_activity"] = station_profile.sum(
    axis=1,
    numeric_only=True
)

station_profile = station_profile.reset_index()

station_profile["morning_net_alighting"] = (
    station_profile["morning_alightings"]
    - station_profile["morning_boardings"]
)

station_profile["evening_net_boarding"] = (
    station_profile["evening_boardings"]
    - station_profile["evening_alightings"]
)

station_profile["midday_share_pct"] = (
    100
    * station_profile["midday_activity"]
    / station_profile["total_weekday_activity"]
)

#  Apply classification logic

In [21]:
import numpy as np
import pandas as pd

# Remove duplicate columns left over from earlier pivot/merge attempts.
station_profile = station_profile.loc[
    :, ~station_profile.columns.duplicated()
].copy()

# Confirm there is exactly one version of every classification field.
required_columns = [
    "total_weekday_activity",
    "morning_net_alighting",
    "evening_net_boarding",
    "midday_share_pct"
]

missing_columns = [
    column for column in required_columns
    if column not in station_profile.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Force all classification measures to numeric values.
for column in required_columns:
    station_profile[column] = pd.to_numeric(
        station_profile[column],
        errors="coerce"
    ).fillna(0)

# Create thresholds separately for each year.
# This avoids comparing full-year 2024 totals directly with partial-year 2025 totals.
station_profile["high_volume_cutoff"] = (
    station_profile
    .groupby("source_year")["total_weekday_activity"]
    .transform(lambda x: x.quantile(0.75))
)

station_profile["very_high_volume_cutoff"] = (
    station_profile
    .groupby("source_year")["total_weekday_activity"]
    .transform(lambda x: x.quantile(0.90))
)

station_profile["commercial_cutoff"] = (
    station_profile
    .groupby("source_year")["midday_share_pct"]
    .transform(lambda x: x.quantile(0.75))
)

# Create Boolean classification conditions.
commuter_condition = (
    (station_profile["morning_net_alighting"] > 0)
    & (station_profile["evening_net_boarding"] > 0)
    & (
        station_profile["total_weekday_activity"]
        >= station_profile["high_volume_cutoff"]
    )
)

residential_condition = (
    (station_profile["morning_net_alighting"] < 0)
    & (station_profile["evening_net_boarding"] < 0)
    & (
        station_profile["total_weekday_activity"]
        >= station_profile["high_volume_cutoff"]
    )
)

commercial_condition = (
    (
        station_profile["midday_share_pct"]
        >= station_profile["commercial_cutoff"]
    )
    & (
        station_profile["morning_net_alighting"].abs()
        < 0.10 * station_profile["total_weekday_activity"]
    )
)

mixed_condition = (
    station_profile["total_weekday_activity"]
    >= station_profile["very_high_volume_cutoff"]
)

# Apply classification without using .apply().
station_profile["station_function"] = np.select(
    [
        commuter_condition,
        residential_condition,
        commercial_condition,
        mixed_condition
    ],
    [
        "Commuter / employment hub",
        "Residential origin hub",
        "Commercial / leisure hub",
        "High-volume mixed hub"
    ],
    default="Local / balanced station"
)

station_profile[
    [
        "source_year",
        "station_name",
        "line_name",
        "total_weekday_activity",
        "morning_net_alighting",
        "evening_net_boarding",
        "midday_share_pct",
        "station_function"
    ]
].head()

direction,source_year,station_name,line_name,total_weekday_activity,morning_net_alighting,evening_net_boarding,midday_share_pct,station_function
hour_bucket,,,,,,,,
0,2024,서울역,1호선,51938431,3081716,3056390,15.144639,Commuter / employment hub
1,2024,시청,1호선,26545110,3097958,2740455,13.022210,Commuter / employment hub
2,2024,종각,1호선,38802055,4512759,3823715,12.415698,Commuter / employment hub
3,2024,종로3가,1호선,23136999,967231,1297967,20.214078,Commuter / employment hub
4,2024,종로5가,1호선,23390351,1520926,1565417,20.105363,Commuter / employment hub


In [22]:
station_profile["station_function"].value_counts()

station_function
Local / balanced station     307
Commercial / leisure hub     108
Commuter / employment hub     73
Residential origin hub        57
High-volume mixed hub          2
Name: count, dtype: int64

In [23]:
print(station_profile.columns.tolist())

print(
    "Duplicate columns:",
    station_profile.columns[
        station_profile.columns.duplicated()
    ].tolist()
)

print(station_profile[required_columns].dtypes)

[('source_year', ''), ('line_name', ''), ('station_code', ''), ('station_name', ''), ('alight', '06_07'), ('alight', '07_08'), ('alight', '08_09'), ('alight', '09_10'), ('alight', '10_11'), ('alight', '11_12'), ('alight', '12_13'), ('alight', '13_14'), ('alight', '14_15'), ('alight', '15_16'), ('alight', '16_17'), ('alight', '17_18'), ('alight', '18_19'), ('alight', '19_20'), ('alight', '20_21'), ('alight', '21_22'), ('alight', '22_23'), ('alight', '23_24'), ('alight', 'after_24'), ('alight', 'before_06'), ('board', '06_07'), ('board', '07_08'), ('board', '08_09'), ('board', '09_10'), ('board', '10_11'), ('board', '11_12'), ('board', '12_13'), ('board', '13_14'), ('board', '14_15'), ('board', '15_16'), ('board', '16_17'), ('board', '17_18'), ('board', '18_19'), ('board', '19_20'), ('board', '20_21'), ('board', '21_22'), ('board', '22_23'), ('board', '23_24'), ('board', 'after_24'), ('board', 'before_06'), ('morning_boardings', ''), ('morning_alightings', ''), ('evening_boardings', ''),

#  YoY comparison

In [30]:
comparable = daily[
    daily["month"].between(7, 12)
].copy()

# Calculate station-level ridership for each source year.
yoy = (
    comparable
    .groupby(
        [
            "source_year",
            "line_name",
            "station_code",
            "station_name"
        ],
        as_index=False
    )["total_activity"]
    .sum()
    .pivot_table(
        index=[
            "line_name",
            "station_code",
            "station_name"
        ],
        columns="source_year",
        values="total_activity",
        aggfunc="sum"
    )
    .reset_index()
)

# Keep only stations present in both comparable periods.
yoy = yoy.dropna(subset=[2024, 2025]).copy()

# Exclude low-volume stations to prevent misleading percentage changes.
yoy = yoy[yoy[2024] >= 100_000].copy()

# Calculate percentage change from Jul–Dec 2024 to Jul–Dec 2025.
yoy["yoy_jul_dec_change_pct"] = (
    100
    * (yoy[2025] - yoy[2024])
    / yoy[2024]
)

# Optional: inspect the highest-growth stations.
yoy.sort_values(
    "yoy_jul_dec_change_pct",
    ascending=False
).head(10)

source_year,line_name,station_code,station_name,2024,2025,yoy_jul_dec_change_pct
258,8호선,2810,암사역사공원,1495597.0,2257136.0,50.918730
165,5호선,2555,둔촌동,3249595.0,4656857.0,43.305766
0,1호선,150,서울역,19426706.0,26465992.0,36.235098
78,3호선,328,잠원,1782295.0,2380503.0,33.563916
125,5호선,2515,마곡,4048355.0,5242493.0,29.496870
116,4호선,430,이촌(국립중앙박물관),3499292.0,4406952.0,25.938390
20,2호선,211,성수,16729048.0,19987146.0,19.475693
210,6호선,2644,돌곶이,3459037.0,4045960.0,16.967815
117,4호선,431,동작(현충원),736339.0,860762.0,16.897516
22,2호선,213,구의(광진구청),8685237.0,9712797.0,11.831111


# Build final evidence table

In [33]:
# Fix MultiIndex columns created by an earlier pivot_table operation.

if isinstance(station_profile.columns, pd.MultiIndex):
    station_profile.columns = [
        "_".join(
            str(value)
            for value in column
            if value not in ("", None)
        )
        for column in station_profile.columns
    ]

# Remove any duplicate columns from earlier notebook attempts.
station_profile = station_profile.loc[
    :, ~station_profile.columns.duplicated()
].copy()

# Check that columns are now one-dimensional.
print(station_profile.columns.tolist())

['source_year', 'line_name', 'station_code', 'station_name', 'alight_06_07', 'alight_07_08', 'alight_08_09', 'alight_09_10', 'alight_10_11', 'alight_11_12', 'alight_12_13', 'alight_13_14', 'alight_14_15', 'alight_15_16', 'alight_16_17', 'alight_17_18', 'alight_18_19', 'alight_19_20', 'alight_20_21', 'alight_21_22', 'alight_22_23', 'alight_23_24', 'alight_after_24', 'alight_before_06', 'board_06_07', 'board_07_08', 'board_08_09', 'board_09_10', 'board_10_11', 'board_11_12', 'board_12_13', 'board_13_14', 'board_14_15', 'board_15_16', 'board_16_17', 'board_17_18', 'board_18_19', 'board_19_20', 'board_20_21', 'board_21_22', 'board_22_23', 'board_23_24', 'board_after_24', 'board_before_06', 'morning_boardings', 'morning_alightings', 'evening_boardings', 'evening_alightings', 'midday_activity', 'total_weekday_activity', 'morning_net_alighting', 'evening_net_boarding', 'midday_share_pct', 'high_volume_cutoff', 'very_high_volume_cutoff', 'commercial_cutoff', 'station_function']


In [34]:
station_profile["station_code"] = (
    station_profile["station_code"].astype(str).str.strip()
)

yoy["station_code"] = (
    yoy["station_code"].astype(str).str.strip()
)

station_profile["line_name"] = (
    station_profile["line_name"].astype(str).str.strip()
)

yoy["line_name"] = (
    yoy["line_name"].astype(str).str.strip()
)

station_profile["station_name"] = (
    station_profile["station_name"].astype(str).str.strip()
)

yoy["station_name"] = (
    yoy["station_name"].astype(str).str.strip()
)

yoy_for_merge = yoy[
    [
        "line_name",
        "station_code",
        "station_name",
        "yoy_jul_dec_change_pct"
    ]
].drop_duplicates()

evidence_table = station_profile.merge(
    yoy_for_merge,
    on=["line_name", "station_code", "station_name"],
    how="left",
    validate="many_to_one"
)